### ML PROJECT EDA

Our project objective is to assess whether company news events and sentiment improve short-term stock return predictions beyond traditional financial characteristics. We will focus on companies in the S&P 500 and compare models using traditional predictors alone with models that also incorporate information from RavenPack.

The main question we want to answer is whether the type of news event, its sentiment, relevance, and novelty can help predict stock returns after the news is released. 

Here is the stock data only

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [5]:
df = pd.read_csv('data/ewwi65k5su2a3ly7.csv')

/var/folders/h4/sj485mdx17dbbf063mnvxn640000gn/T/ipykernel_86186/2103063123.py:1: DtypeWarning: Columns (24: DlyRetMissFlg) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('data/ewwi65k5su2a3ly7.csv')


In [6]:
df.head(2)

,INDNO,MbrStartDt,MbrEndDt,PERMNO,HdrCUSIP,PERMCO,SICCD,CUSIP,Ticker,ShareType,...,DlyLow,DlyHigh,DlyBid,DlyAsk,DlyOpen,DlyNumTrd,DlyMMCnt,DlyCumFacPr,DlyCumFacShr,ShrOut
0,1000500,1989-08-03,2025-12-31,10104,68389X10,8045,7379,68389X10,ORCL,NS,...,44.45,45.34,45.20,45.21,44.48,NaN,NaN,1.0,1.0,3588919.0
1,1000500,1994-06-07,2025-12-31,10107,59491810,8048,7372,59491810,MSFT,NS,...,98.94,101.75,101.12,101.13,99.55,239618.0,66.0,1.0,1.0,7683000.0


In [7]:
print ("Observations:", len(df))
print ("Variables:", len(df.columns))
print ("Unique PERMNOs:", len(df["PERMNO"].unique()))
print ("Unique Tickers:", len(df["Ticker"].unique()))
print("Number of trading days:", len(df['DlyCalDt'].unique()))

Observations: 761226
Variables: 42
Unique PERMNOs: 617
Unique Tickers: 646
Number of trading days: 1510


In [8]:
df.select_dtypes(exclude='float').dtypes

INDNO               int64
MbrStartDt            str
MbrEndDt              str
PERMNO              int64
HdrCUSIP              str
PERMCO              int64
SICCD               int64
CUSIP                 str
Ticker                str
ShareType             str
SecurityType          str
SecuritySubType       str
USIncFlg              str
IssuerType            str
PrimaryExch           str
ConditionalType       str
TradingStatusFlg      str
DlyCalDt              str
DlyPrcFlg             str
DlyCapFlg             str
DlyRetMissFlg         str
DlyRetDurFlg          str
DlyDistRetFlg         str
dtype: object

In [9]:
def display_nan_val(df,threshold):
    nan_percent = df.isna().mean() * 100
    nan_cols = nan_percent[nan_percent>threshold].index.tolist()
    return nan_percent[nan_percent>threshold] 

In [10]:
print('*'*40)
print ('COLUMNS has more than 50% nan')
print('*'*40)
display_nan_val(df,threshold=50)

****************************************
COLUMNS has more than 50% nan
****************************************


DlyRetMissFlg    99.996322
DlyNumTrd        71.119615
DlyMMCnt         71.119615
dtype: float64

In [11]:
# Miss value in 'DlyNumTrd', 'DlyMMCnt' is associated with the exchange (PrimaryExch)
nancols = ['DlyNumTrd', 'DlyMMCnt']
df.groupby('PrimaryExch')[nancols].apply(
    lambda x: x.isna().mean() * 100
)

,DlyNumTrd,DlyMMCnt
PrimaryExch,,
B,100.0,100.0
N,100.0,100.0
Q,0.0,0.0
X,100.0,100.0


#### 1. Data Cleaning
There are approximately 1,510 trading days between 2019-01-01 till 2024-12-31. Filter the dataset to include only the stocks that remained in the S&P 500 index continuously throughout the entire period, removing any constituents that were added or removed temporarily

1. Convert DlyCalDt into datetime
2. Sort observations by PERMNO and date
3. Check duplicate
4. Drop stocks that were in the S&P 500 temporarily
5. Drop columns that only has 1 values
6. Drop 'DlyRetMissFlg','DlyMMCnt','DlyNumTrd', too much missing values and DlyNumTrd is dep on exchanges
7. Investigate missing values and fillna

In [275]:
def data_cleaning(df):
    df = df.copy()
    df['DlyCalDt'] = pd.to_datetime(df['DlyCalDt'])
    df = df.sort_values(by=['PERMNO', 'DlyCalDt']).reset_index(drop=True)
    assert df.duplicated(subset=['PERMNO', 'DlyCalDt']).sum()==0 #check duplicates
    ## only keep stocks that cover the whole period  
    dcal = df['PERMNO'].value_counts()
    stock_ids = dcal[dcal==1510].index
    df = df[df['PERMNO'].isin(stock_ids)]

    #drop columns that only has 1 values
    drop_cols = [col for col in df.columns if df[col].nunique() == 1]
    drop_cols+=['HdrCUSIP','MbrStartDt','MbrEndDt','DlyRetMissFlg','DlyMMCnt','DlyNumTrd']
    df = df.drop(columns=drop_cols)
    print ("Observations:", len(df))
    print ("Variables:", len(df.columns))
    return df
df_filter = data_cleaning(df)

Observations: 614570
Variables: 31


According to OCC, Biogen Inc. (BIIB) is missing trading data on 2020-11-06 and 2023-06-09 because Nasdaq halted trading in BIIB shares for the entire day. we will use forward filling to treat this missing value

In [276]:
def treat_na(df):
    small_nan_cols = df.columns[df.isnull().any()].difference(['DlyRetMissFlg','DlyNumTrd', 'DlyMMCnt']).to_list()
    dn = df[df[small_nan_cols].isna().any(axis=1)]
    display(dn)
    ### patch for BIIB 
    m = df['Ticker'].eq('BIIB')
    df.loc[m, ['DlyVol']] = df.loc[m, ['DlyVol']].fillna(0)
    df.loc[m] = df.loc[m].ffill()
    return df

df_filter = treat_na(df_filter)

,PERMNO,PERMCO,SICCD,CUSIP,Ticker,ShareType,USIncFlg,IssuerType,PrimaryExch,DlyCalDt,...,DlyVol,DlyClose,DlyLow,DlyHigh,DlyBid,DlyAsk,DlyOpen,DlyCumFacPr,DlyCumFacShr,ShrOut
483351,76841,11112,2836,09062X10,BIIB,NS,Y,CORP,Q,2020-11-06,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,153882.0
484001,76841,11112,2836,09062X10,BIIB,NS,Y,CORP,Q,2023-06-09,...,0.0,NaN,NaN,NaN,303.8,330.0,NaN,1.0,1.0,144742.0


#### 2. Data Featuring
1. Returns Y, shift down by 1 day
2. spread bid-ask
3. Momentum features
4. Volatility daily and rolling
5. Market capitalization and in log scale: **market cap = share price x shares outstanding**
6. Liquidity volumn in log scale
7. Liquidity turnover Volume / shares outstanding

In [370]:
def data_featuring(df):
    df = df.copy()
    g = df.groupby('PERMNO')
    df['LogRet'] = np.log1p(df['DlyRet'])
    df['RetLead1m'] = df['DlyRet'].shift(-1) ### Y target 
    df['mom5']=  g['LogRet'].transform(lambda s: s.rolling(5).sum())
    df['mom21']=  g['LogRet'].transform(lambda s: s.rolling(21).sum())
    df['vol5'] = g['LogRet'].transform(lambda s: s.rolling(3, min_periods=3).std()) * np.sqrt(252)
    df['vol10'] = g['LogRet'].transform(lambda s: s.rolling(21, min_periods=10).std()) * np.sqrt(252)
    df['spread'] = (df['DlyAsk']-df['DlyBid'])/(df['DlyBid']+df['DlyAsk'])/2  #(Ask - Bid) / ((Ask + Bid) / 2
    df['ShrOut'] = df['ShrOut'] * df['DlyCumFacShr'] #drop cumfacshr after use
    df['mktcap'] = np.log10(df['DlyClose']*df['ShrOut'])

    # df = df.drop(columns=['DlyCumFacShr'])
    print (df.loc[df['Ticker']=='CBOE',['Ticker','DlyCalDt','DlyClose','DlyRet','RetLead1m','mom5']].head(10))
    return

data_featuring(df_filter)

       Ticker   DlyCalDt  DlyClose    DlyRet  RetLead1m      mom5
758703   CBOE 2019-01-02     95.89 -0.019830  -0.003129       NaN
758704   CBOE 2019-01-03     95.59 -0.003129   0.028978       NaN
758705   CBOE 2019-01-04     98.36  0.028978  -0.018402       NaN
758706   CBOE 2019-01-07     96.55 -0.018402  -0.031383       NaN
758707   CBOE 2019-01-08     93.52 -0.031383  -0.007699 -0.045057
758708   CBOE 2019-01-09     92.80 -0.007699  -0.013470 -0.032756
758709   CBOE 2019-01-10     91.55 -0.013470  -0.001748 -0.043184
758710   CBOE 2019-01-11     91.39 -0.001748   0.008863 -0.073499
758711   CBOE 2019-01-14     92.20  0.008863   0.005640 -0.046102
758712   CBOE 2019-01-15     92.72  0.005640   0.002696 -0.008592


Dealing with missing values

In [88]:
df_filtered['USIncFlg'].value_counts()

USIncFlg
Y    581350
N     33220
Name: count, dtype: int64